In [ ]:
# ============================================================
# Notebook 01: "Where Do Coefficients Come From?"
# Regression from the Inside: Seeing the Geometry of Linear Models
# ============================================================

# --- Environment setup (run this cell first) ---
import sys

# Install regression_geometry package if not available
try:
    import regression_geometry
except ImportError:
    # Running in Colab or fresh environment — install from GitHub
    print("Installing regression_geometry package...")
    !pip install -q git+https://github.com/carloscotrini/sml_lab26_geometric_regression.git
    print("Done! If you see import errors below, restart the runtime (Runtime → Restart) and run this cell again.")

# --- Standard imports ---
import numpy as np
import matplotlib.pyplot as plt
from scipy import linalg

import statsmodels.api as sm

from regression_geometry.core import ColumnSpace, Projection
from regression_geometry.data import load_meridian
from regression_geometry.colors import *

# --- Rendering backend toggle ---
INTERACTIVE = True
try:
    from regression_geometry import interactive as viz_mod
    if not viz_mod.AVAILABLE:
        INTERACTIVE = False
except ImportError:
    INTERACTIVE = False

if INTERACTIVE:
    from regression_geometry import interactive as viz
else:
    from regression_geometry import plots as viz

from regression_geometry.scoreboard import GeometricScoreboard

# --- Reproducibility ---
np.random.seed(42)

> *"Every formula is a fossil — it remembers the shape of the idea that made it."*

## Where Do Coefficients Come From?

You've probably run hundreds of regressions. Every time, the computer hands you a coefficient — 2.14, 0.73, -0.08.

But where do those numbers come from? Not "what formula does the computer use" — you can look that up. The real question: what IS that number, geometrically? What shape does it have?

This notebook answers that question.

In [ ]:
# Generate 15 data points and fit a regression
rng = np.random.default_rng(42)
x = rng.uniform(0, 10, size=15)
y_true = 2 * x + 3
y = y_true + rng.normal(0, 2, size=15)

X = sm.add_constant(x)
model = sm.OLS(y, X).fit()
print(f"Intercept: {model.params[0]:.2f}")
print(f"Slope:     {model.params[1]:.2f}")
print(f"R\u00b2:        {model.rsquared:.3f}")
print(f"\nThese numbers appeared in milliseconds. But what just happened?")

## Try Your Own Line

Before asking the computer, try it yourself. Pick a slope and intercept. How good is your line?

In [ ]:
# Try your own slope and intercept
my_slope = 2.0      # ← change this
my_intercept = 3.0   # ← change this

y_pred = my_slope * x + my_intercept
residuals = y - y_pred
ssr = np.sum(residuals**2)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(x, y, color=RESPONSE_Y, s=60, zorder=3, label="Data")
xs = np.linspace(0, 10, 100)
ax.plot(xs, my_slope * xs + my_intercept, color=PROJECTION, lw=2, label="Your line")
for xi, yi, ypi in zip(x, y, y_pred):
    ax.plot([xi, xi], [yi, ypi], color=RESIDUAL, lw=1, alpha=0.6)
ax.set(title=f"Your line (SSR = {ssr:.1f})", xlabel="x", ylabel="y")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Compute SSR across a grid of slopes and intercepts
slopes = np.linspace(0, 4, 80)
intercepts = np.linspace(-2, 8, 80)
S, I = np.meshgrid(slopes, intercepts)
SSR = np.array([[np.sum((y - (s*x + i))**2) for s in slopes] for i in intercepts])

fig, ax = plt.subplots(figsize=(8, 6))
cs_plot = ax.contour(S, I, SSR, levels=20, cmap="Blues")
ax.clabel(cs_plot, inline=True, fontsize=8)
ax.plot(my_slope, my_intercept, "o", color=RESIDUAL, ms=10, label="Your guess")
ax.plot(model.params[1], model.params[0], "*", color=PROJECTION, ms=15, label="OLS")
ax.set(xlabel="Slope", ylabel="Intercept", title="Loss Landscape")
ax.legend()
plt.tight_layout()
plt.show()

---
### 🛑 PREDICT FIRST

We're about to look at this same problem from a completely different angle — literally. Instead of a scatter plot, we'll treat the data as a single object in a high-dimensional space.

**Before running the next cell, write your prediction below:**

Why does the formula β̂ = (X'X)⁻¹X'y give you the bottom of the bowl? What is that formula actually DOING?

---

In [ ]:
# YOUR PREDICTION:
#
#

## From Scatter Plot to Vector

Here's the idea that changes everything. Forget the scatter plot. Instead of thinking of 15 separate data points, think of all 15 y-values as a single object — a list of 15 numbers. That's one vector.

If you only had 2 data points, your y-vector would be [y₁, y₂] — just two numbers. That's a point on a flat 2D plane.

In [ ]:
# With 2 data points, y is a point in 2D — easy to see
x2 = np.array([2.0, 8.0])
y2 = np.array([6.5, 19.2])

fig, ax = plt.subplots(figsize=(6, 6))
ax.arrow(0, 0, y2[0], y2[1], head_width=0.3, head_length=0.2,
         fc=RESPONSE_Y, ec=RESPONSE_Y, linewidth=2)
ax.text(y2[0]+0.3, y2[1], f"y = [{y2[0]}, {y2[1]}]", fontsize=12, color=RESPONSE_Y)
ax.set(xlim=(-1, 22), ylim=(-1, 22), xlabel="$y_1$ axis", ylabel="$y_2$ axis",
       title="y as a vector in $\\mathbb{R}^2$")
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Three Data Points: Now You Can See the Geometry

With 3 data points, y = [y₁, y₂, y₃] lives in 3D space. And the set of ALL possible predictions from a linear model forms a flat plane through the origin. Every point on this plane corresponds to some choice of slope and intercept.

Think of it as a wall. The gold arrow (your data) floats in space. The blue plane (all possible predictions) is the wall.

In [ ]:
# Build the 3D picture: y (gold), prediction surface (blue), y_hat (green), e (red)
x3 = np.array([2.0, 5.0, 8.0])
y3 = np.array([6.5, 12.8, 19.2])

# ColumnSpace handles the math
cs3 = ColumnSpace(x3, add_intercept=True)  # adds column of 1s

# Use the viz module for the canonical 3D plot
fig = viz.plot_projection_3d(cs3, y3, title="The closest point falls at a right angle")

The red arrow hits the blue plane at a right angle. That's not a coincidence — that's the definition of OLS.

The regression's prediction — ŷ — is the point on the plane that is CLOSEST to y. And the shortest path from a point to a plane is always perpendicular.

In [ ]:
# Verify: the residual is perpendicular to every column of X
proj3 = cs3.project(y3)
e3 = proj3.residuals
X3 = cs3.X  # the design matrix with intercept

print(f"Dot product of e with intercept column: {X3[:, 0] @ e3:.10f}")
print(f"Dot product of e with x column:         {X3[:, 1] @ e3:.10f}")
print()
print("Both are zero. The residual is perpendicular to every direction on the plane.")
print("This is what the equation X'e = 0 means — and now you can SEE it.")

In [ ]:
# Interactive 3D exploration — try different y vectors
fig = viz.plot_projection_3d(cs3, y3, views="three_panel",
                             title="Three views of the same perpendicular")

## The Flashlight Metaphor

Think of it this way. Imagine y is an object hanging in space above the blue plane. A flashlight shines straight down — perpendicular to the plane. The shadow of y on the plane is ŷ.

Your prediction is a shadow. The regression line is a shadow. The residual is the distance from the object to its shadow.

And the shadow always falls at a right angle.

---
### 🛑 PREDICT FIRST

If you add a second predictor — say, x₂ — what happens to the blue plane? Does it stay the same size? Expand? Tilt?

**Before running the next cell, write your prediction below:**

What happens to the residual — does it get longer or shorter? What happens to R²?

---

In [ ]:
# YOUR PREDICTION:
#
#

In [ ]:
# Add a second predictor — compare to your prediction above
x3_b = np.array([1.0, 3.0, 2.0])
X3_full = np.column_stack([np.ones(3), x3, x3_b])
cs3_full = ColumnSpace(X3_full, add_intercept=False)  # already has intercept

proj3_one = cs3.project(y3)
proj3_two = cs3_full.project(y3)

print(f"With 1 predictor:  R\u00b2 = {proj3_one.r_squared:.4f}, ||e|| = {proj3_one.residual_norm:.4f}")
print(f"With 2 predictors: R\u00b2 = {proj3_two.r_squared:.4f}, ||e|| = {proj3_two.residual_norm:.4f}")
print()
print("Adding a predictor expanded the plane. The shadow got closer.")
print("This is why adding variables always increases R\u00b2 —")
print("you're projecting onto a bigger surface.")

## The Formula Is a Recipe for Dropping a Perpendicular

In [ ]:
# Show that beta = (X'X)^-1 X'y computes the same thing
X3_mat = cs3.X
XtX = X3_mat.T @ X3_mat
Xty = X3_mat.T @ y3
beta = np.linalg.solve(XtX, Xty)

print("Step by step:")
print(f"  X'X =\n{XtX}")
print(f"  X'y = {Xty}")
print(f"  beta = (X'X)^-1 X'y = {beta}")
print(f"\n  y_hat from formula:   {X3_mat @ beta}")
print(f"  y_hat from ColumnSpace: {proj3.y_hat}")
print(f"\nSame answer. The formula is a recipe for dropping a perpendicular.")

In [ ]:
# The same works in R^15 — verify on the original n=15 data
e_full = model.resid
ortho = sm.add_constant(x).T @ e_full

print(f"X'e = {ortho}")
print(f"Both zero. The perpendicular holds in 15 dimensions.")
print(f"\nCoefficients match: intercept={model.params[0]:.4f}, slope={model.params[1]:.4f}")
print(f"\nYou can't draw R^15, but you can KNOW the right angle is there.")

## First Encounter: Meridian Analytics

In [ ]:
# First encounter with Meridian Analytics
df = load_meridian()
X_mer = sm.add_constant(df["experience"].values)
model_mer = sm.OLS(df["salary"].values, X_mer).fit()
print(f"Each year of experience: ${model_mer.params[1]:,.0f}")
print(f"R\u00b2: {model_mer.rsquared:.3f}")
print(f"Employees: {len(df)}")

Meridian Analytics has 2,000 employees. Each year of experience is associated with additional salary. Somewhere in ℝ²⁰⁰⁰, there's a gold arrow being projected onto a blue plane, and the foot of the perpendicular gives us that number.

You'll return to Meridian throughout this course.

---
### 🛑 PREDICT FIRST

You just saw that experience alone gives a certain R². If you add education as a second predictor, what will happen?

**Before running the next cell, write your prediction below:**

Will R² go up, go down, or stay the same? By a little or a lot?

---

In [ ]:
# YOUR PREDICTION:
#
#

In [ ]:
# Compare to your prediction: add education
X_mer2 = sm.add_constant(df[["experience", "education"]])
model_mer2 = sm.OLS(df["salary"], X_mer2).fit()
print(f"salary ~ experience:               R\u00b2 = {model_mer.rsquared:.3f}")
print(f"salary ~ experience + education:   R\u00b2 = {model_mer2.rsquared:.3f}")
print(f"\nR\u00b2 went up. A bigger plane catches more of y.")

---
### 🛑 DIAGNOSE FIRST

Look at the R² from the Meridian simple regression above.

**Before seeing the visualization, answer:**

What does that R² value mean geometrically, in terms of the gold arrow, the green shadow, and the red residual?

---

In [ ]:
# YOUR ANSWER:
#
#

---
### 🔗 Back to Statsmodels

| Geometric concept | Code |
|---|---|
| Projection ŷ (shadow on the plane) | `model.fittedvalues` |
| Residual e (object minus shadow) | `model.resid` |
| Perpendicularity check X'e = 0 | `X.T @ model.resid` |
| Coefficients (coordinates of the shadow) | `model.params` |

---

In [ ]:
# Demonstrate each mapping on the Meridian model
print("Projection (first 5 fitted values):")
print(f"  {model_mer.fittedvalues[:5].round(0)}")
print(f"\nResidual (first 5):")
print(f"  {model_mer.resid[:5].round(0)}")
print(f"\nPerpendicularity check (X'e):")
print(f"  {(X_mer.T @ model_mer.resid).round(8)}")
print(f"  Both zero — the residual is perpendicular to the plane.")

---
### ✍️ The Memo

You're writing a memo to Meridian's VP of HR.

In 3 sentences, explain what the regression coefficient for experience means — both what the number says and how the computer found it.

**Rules:** Do not use the words "projection," "vector," "perpendicular," or "column space." Write in plain English that a non-technical manager would understand.

---

In [ ]:
# YOUR MEMO:
#
#
#

In [ ]:
# Geometric Scoreboard — first appearance
cs_mer = ColumnSpace(X_mer, add_intercept=False)  # X_mer already has intercept
proj_mer = cs_mer.project(df["salary"].values)

scoreboard = GeometricScoreboard(
    proj=proj_mer,
    cs=cs_mer,
    active_gauges=["theta", "r_squared"],
    mode="widget" if INTERACTIVE else "static"
)
scoreboard.display()

These five numbers are your instruments. You can't read most of them yet. By Notebook 9, you'll glance at these gauges and diagnose any regression instantly.

---
### Summary

**What you learned:**
- Regression coefficients are coordinates of a shadow — the projection of the data vector y onto the column space of X.
- The fitted values ŷ are the closest point to y on the column space plane.
- The residual e is always perpendicular to that plane: X'e = 0.

**Key geometric insight:** ***β̂ is the foot of the perpendicular.***

### Next

The perpendicularity isn't just a nice picture — it's the most important property in statistics. Notebook 2 asks: what does the right angle guarantee, and what can go wrong when it breaks?

---